In [1]:
import Pkg

In [12]:
Pkg.activate(".")
Pkg.add("AppleAccelerate")
Pkg.add("ThreadPinning")
Pkg.add("BenchmarkTools")
Pkg.add("ProfileCanvas")
Pkg.add("QuantumControl")
Pkg.add("QuantumPropagators")
Pkg.add("QuantumControlTestUtils")
Pkg.instantiate()

  Activating project at `~/Documents/Research/2025-05_KrylovKitBenchmark`
   Resolving package versions...
    Updating `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  [13e28ba4] + AppleAccelerate v0.4.0
    Updating `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
  [13e28ba4] + AppleAccelerate v0.4.0
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Project.toml`
  No Changes to `~/Documents/Research/2025-05_KrylovKitBenchmark/Manifest.toml`
   Resolving package versions...
  No Changes to `~/Documents/Research/2025-05_KrylovKit

In [3]:
using ThreadPinning
pinthreads(:cores)
threadinfo()

Hostname: 	ophelia
CPU(s): 	1 x Apple M4 Max
CPU target: 	apple-m1
Cores: 		16 (16 CPU-threads)
Core kinds: 	4 "efficiency cores", 12 "performance cores".
NUMA domains: 	1 (16 cores each)

Unsupported OS: Won't be able to highlight Julia threads.

Julia threads: 	8

CPU socket 1
  0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15


# = Julia thread, # = >1 Julia thread, # = Efficiency core


In [4]:
filter(p -> contains(p[1], "THREAD"), ENV)

Dict{String, String} with 6 entries:
  "OPENBLAS_NUM_THREADS"   => "1"
  "VECLIB_MAXIMUM_THREADS" => "1"
  "OMP_NUM_THREADS"        => "1"
  "NUMEXPR_NUM_THREADS"    => "1"
  "MKL_NUM_THREADS"        => "1"
  "JULIA_NUM_THREADS"      => "8"

In [5]:
using QuantumControl: Trajectory
using QuantumControlTestUtils.DummyOptimization: dummy_control_problem

In [13]:
using AppleAccelerate

In [14]:
N = 100

100

In [15]:
problem = dummy_control_problem(; N, n_trajectories=Threads.nthreads(), n_steps=100, density=1.0, hermitian=true)
trajs = problem.trajectories;
tlist = problem.tlist;

In [16]:
using QuantumPropagators: Cheby
using QuantumControl: propagate_trajectories

In [17]:
propagate_trajectories(trajs, tlist; method=Cheby, use_threads=true);

In [18]:
using BenchmarkTools

In [19]:
bm_sequential = @benchmark propagate_trajectories($trajs, $tlist; method=Cheby, use_threads=false)

BenchmarkTools.Trial: 147 samples with 1 evaluation.
 Range (min … max):  33.293 ms …  39.845 ms  ┊ GC (min … max): 0.00% … 16.07%
 Time  (median):     33.945 ms               ┊ GC (median):    1.86%
 Time  (mean ± σ):   34.206 ms ± 810.519 μs  ┊ GC (mean ± σ):  2.41% ±  1.93%

        ▄▄▆ █                                                   
  ▄▅▄▄▆████▇█▇▄▅▆▅▅▄▆▆▇▁▃▆▇▅▇▄▁▁▃▃▁▄▅▃▄▁▁▁▄▃▁▇▁▁▁▁▃▃▁▁▁▃▃▃▃▁▁▃ ▃
  33.3 ms         Histogram: frequency by time         36.2 ms <

 Memory estimate: 18.72 MiB, allocs estimate: 25672.

In [20]:
bm_parallel = @benchmark propagate_trajectories($trajs, $tlist; method=Cheby, use_threads=true)

BenchmarkTools.Trial: 620 samples with 1 evaluation.
 Range (min … max):  6.736 ms …  12.216 ms  ┊ GC (min … max):  0.00% … 41.48%
 Time  (median):     8.082 ms               ┊ GC (median):    11.33%
 Time  (mean ± σ):   8.067 ms ± 766.165 μs  ┊ GC (mean ± σ):  11.04% ±  7.51%

    ▁▄▂▂▇▁▂▂▁        ▅▃▅█▁▇▆▄▄▅▆▆▂ ▅▃▂▄        ▁               
  ▃▄█████████▅▄▄▅▃▅████████████████████▇▅▅▇█▅▄▅██▅▃▃▃▃▃▄▁▄▄▁▃ ▅
  6.74 ms         Histogram: frequency by time        9.87 ms <

 Memory estimate: 18.73 MiB, allocs estimate: 25714.

In [21]:
mean(bm_sequential.times) / mean(bm_parallel.times)

4.240376502756066